In [ ]:
from multimodal_jupy_logger import (
    print_environment_check,
)

REQUIRED_PACKAGES = {
    "IPython": "ipython",
    "matplotlib": "matplotlib",
}

OPTIONAL_PACKAGES = {
    "nbformat": "nbformat",
    "nbconvert": "nbconvert",
    "traitlets": "traitlets",
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "torch": "torch",
    "tensorflow": "tensorflow",
}

environment_report = print_environment_check(
    required_packages=REQUIRED_PACKAGES,
    investigation_packages=OPTIONAL_PACKAGES,
    show_import_paths=False,
)

if environment_report["required_missing"]:
    install_text = " ".join(
        environment_report["required_missing"]
    )

    raise RuntimeError(
        "Required packages are missing.\n\n"
        f"Run: %pip install {install_text}\n\n"
        "Then restart the kernel."
    )
##endof:  if environment_report["required_missing"]

print()
print("[PASS] Required notebook environment is ready.")


# MMJL portable smoke test

This notebook checks the active environment directly in its first cell. It does not require `environment_checks.py`.

It tests drop-in source discovery, logger registration, quiet magic wrappers, pin-family behavior, relative manifest paths, relative timeline links, validation, and HTML/Markdown export.

For SageMaker, leave:

```python
USE_EXPLICIT_MMJL_SRC = False
```

For a Windows development checkout, set it to `True` and update `EXPLICIT_MMJL_SRC`.

In [ ]:
from pathlib import Path
import importlib
import importlib.metadata
import importlib.util
import os
import platform
import sys

USE_EXPLICIT_MMJL_SRC = False

EXPLICIT_MMJL_SRC = Path(
    r"D:\David\my_repos_dwb\multimodal-jupy-logger\src"
)

REQUIRED_PACKAGES = {
    "IPython": "ipython",
    "matplotlib": "matplotlib",
}

OPTIONAL_NOTEBOOK_TRANSFORMS = {
    "nbformat": "nbformat",
    "nbconvert": "nbconvert",
    "traitlets": "traitlets",
}

OPTIONAL_INVESTIGATION_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "torch": "torch",
    "tensorflow": "tensorflow",
}


def package_version(package_name):
    try:
        return importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        return None
    ##endof:  try/except
##endof:  package_version(...)


def check_packages(title, packages):
    missing_packages = []

    print(title)
    print("-" * len(title))

    for module_name, package_name in packages.items():
        module_spec = importlib.util.find_spec(module_name)
        found = module_spec is not None
        version = package_version(package_name)

        if found:
            version_text = (
                version
                if version is not None
                else "version unknown"
            )

            print(
                f"[FOUND]   {module_name:<16} "
                f"{version_text:<16} "
                f"pip: {package_name}"
            )
        else:
            missing_packages.append(package_name)

            print(
                f"[MISSING] {module_name:<16} "
                f"{'':16} "
                f"pip: {package_name}"
            )
        ##endof:  if found
    ##endof:  for module_name, package_name in packages.items()

    print()

    return missing_packages
##endof:  check_packages(...)


print("=" * 72)
print("MMJL / NOTEBOOK ENVIRONMENT CHECK")
print("=" * 72)
print("Python executable:", sys.executable)
print("Python version:   ", sys.version.replace("\n", " "))
print("Working directory:", Path.cwd())
print("Home directory:   ", Path.home())
print("Platform:         ", platform.platform())
print("Operating system: ", os.name)
print()

required_missing = check_packages(
    "Required packages",
    REQUIRED_PACKAGES,
)

optional_transform_missing = check_packages(
    "Optional notebook/PDF packages",
    OPTIONAL_NOTEBOOK_TRANSFORMS,
)

optional_investigation_missing = check_packages(
    "Optional investigation packages",
    OPTIONAL_INVESTIGATION_PACKAGES,
)

if required_missing:
    install_text = " ".join(required_missing)

    raise RuntimeError(
        "Required packages are missing.\n\n"
        "Install into the active notebook kernel with:\n\n"
        f"  %pip install {install_text}\n\n"
        "Then restart the kernel and rerun this cell."
    )
##endof:  if required_missing

print("[PASS] Required notebook environment is ready.")
print()

starting_dir = Path.cwd().resolve()
candidate_src_paths = []

if USE_EXPLICIT_MMJL_SRC:
    candidate_src_paths.append(
        EXPLICIT_MMJL_SRC.expanduser().resolve()
    )
else:
    for candidate in [starting_dir, *starting_dir.parents]:
        candidate_src_paths.append(candidate / "src")
    ##endof:  for candidate in [...]

    candidate_src_paths.append(
        Path.home() / "multimodal-jupy-logger" / "src"
    )
##endof:  if USE_EXPLICIT_MMJL_SRC

mmjl_src = None

for candidate_src in candidate_src_paths:
    package_dir = (
        candidate_src
        / "multimodal_jupy_logger"
    )

    if package_dir.is_dir():
        mmjl_src = candidate_src
        break
    ##endof:  if package_dir.is_dir()
##endof:  for candidate_src in candidate_src_paths

if mmjl_src is None:
    checked_paths = "\n".join(
        f"  - {candidate}"
        for candidate in candidate_src_paths
    )

    raise RuntimeError(
        "Could not find MMJL source.\n"
        "Checked:\n"
        f"{checked_paths}"
    )
##endof:  if mmjl_src is None

if str(mmjl_src) not in sys.path:
    sys.path.insert(0, str(mmjl_src))
##endof:  if str(mmjl_src) not in sys.path

importlib.invalidate_caches()

print(
    "source mode:",
    (
        "explicit"
        if USE_EXPLICIT_MMJL_SRC
        else "portable discovery"
    ),
)
print("starting_dir:", starting_dir)
print("mmjl_src:", mmjl_src)
print("python:", sys.executable)


In [ ]:
from pathlib import Path
import shutil

from multimodal_jupy_logger import (
    MultimodalJupyLogger,
    register_jupy_logger,
)

log_root = Path.cwd() / "jupy_log_smoke_test"

# Smoke-test root is intentionally disposable.
if log_root.exists():
    shutil.rmtree(log_root)
##endof:  if log_root.exists()

register_jupy_logger(root=log_root)

print("MMJL log root:", log_root)
print("log root exists:", log_root.exists())


In [ ]:
%jupy_inspect

## Literal logging smoke test

Expected:

- no `WindowsPath(...)` or `PosixPath(...)` display after the magic;
- one `[DONE] Logged text: ...` status line;
- a manifest row with a relative `artifacts/...` path.

In [ ]:
%%jupy_log --label smoke-markdown-note --mime text/markdown
# Smoke Markdown note

This text was logged with `%%jupy_log`.

It should appear in the manifest and exported timelines.

## Pin-family smoke tests

Expected:

- `--pin` logs input plus output with an automatic label;
- `--label NAME` with no pin option behaves like `--pin`;
- `--pin-input` logs input only;
- `--pin-output` logs output only.

In [ ]:
%%jupy_capture --pin
pin_message = "pin captures input and output"
print(pin_message)

In [ ]:
%%jupy_capture --label named-pin-default
named_message = "label alone behaves like pin"
print(named_message)

In [ ]:
%%jupy_capture --label input-only-example --pin-input
quiet_value = sum([1, 2, 3, 4])
print("This output should show in Jupyter, but not be logged by --pin-input.")

In [ ]:
%%jupy_capture --label output-only-example --pin-output
output_only_value = 6 * 7
print("This output should be logged, but the input should not be logged.")
output_only_value

## Plot capture smoke test

Expected:

- stdout/display output is logged;
- the Matplotlib display is logged as an image artifact;
- timeline HTML/Markdown uses relative links to the image.

In [ ]:
%%jupy_capture --label tiny-plot --pin
import matplotlib.pyplot as plt

xs = [0, 1, 2, 3, 4]
ys = [x * x for x in xs]

plt.figure(figsize=(5, 3))
plt.plot(xs, ys, marker="o")
plt.title("MMJL smoke test plot")
plt.xlabel("x")
plt.ylabel("x squared")
plt.grid(True)
plt.show()

## Exception capture smoke test

The next cell intentionally raises an exception.

Expected:

- Jupyter shows the error;
- MMJL logs input, stdout, and the exception artifact;
- the kernel remains usable afterward.

Run the next cell intentionally, then manually continue.

In [ ]:
%%jupy_capture --label expected-exception --pin
print("This cell intentionally raises an exception.")
raise ValueError("Expected MMJL smoke-test exception")

## Validate and export

Expected:

- `%jupy_validate` reports zero missing artifacts;
- `%jupy_markdown` and `%jupy_html` create timeline files;
- no stray path/list objects appear as magic output.

In [ ]:
%jupy_inspect
%jupy_validate
%jupy_markdown
%jupy_html

In [ ]:
from pathlib import Path
import csv

manifest_path = log_root / "manifest.tsv"
timeline_md = log_root / "timelines" / "timeline.md"
timeline_html = log_root / "timelines" / "timeline.html"

print("manifest:", manifest_path)
print("timeline_md:", timeline_md)
print("timeline_html:", timeline_html)

assert manifest_path.exists()
assert timeline_md.exists()
assert timeline_html.exists()

rows = list(csv.DictReader(
    manifest_path.open("r", encoding="utf-8", newline=""),
    delimiter="\t",
))

print("manifest rows:", len(rows))

absolute_paths = [
    row["path"]
    for row in rows
    if Path(row["path"]).is_absolute()
]

print("absolute manifest paths:", absolute_paths)

assert not absolute_paths, (
    "Manifest should use relative artifact paths."
)

md_text = timeline_md.read_text(encoding="utf-8")
html_text = timeline_html.read_text(encoding="utf-8")

assert "../artifacts/" in md_text or "../artifacts/" in html_text
assert str(log_root.resolve()) not in md_text
assert str(log_root.resolve()) not in html_text

print("[PASS] Portable relative-path smoke test passed.")

## Optional filesystem view

This final cell uses the external Windows or Linux `tree` command when available. Its failure does not invalidate the Python-based MMJL assertions.

In [ ]:
import os
import shutil
import subprocess

EXCLUDED_TREE_TEXT = (
    "ipynb_checkpoints",
    "checkpoint.ipynb",
    "serial",
)


def _filter_tree_output(output):
    kept_lines = (
        line
        for line in output.splitlines()
        if not any(
            excluded in line
            for excluded in EXCLUDED_TREE_TEXT
        )
    )

    return "\n".join(kept_lines)
##endof:  _filter_tree_output(...)


def _run_tree_command(command):
    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=True,
    )

    return _filter_tree_output(result.stdout)
##endof:  _run_tree_command(...)


def _run_windows_tree():
    if os.name != "nt":
        raise RuntimeError(
            "The Windows tree command requires Windows."
        )
    ##endof:  if os.name != "nt"

    tree_command = shutil.which("tree")

    if tree_command is None:
        raise RuntimeError(
            "The Windows 'tree' command was not found."
        )
    ##endof:  if tree_command is None

    return _run_tree_command(
        [
            tree_command,
            "/a",
            "/f",
            ".",
        ]
    )
##endof:  _run_windows_tree(...)


def _run_linux_tree():
    if os.name == "nt":
        raise RuntimeError(
            "The Linux tree command requires a non-Windows system."
        )
    ##endof:  if os.name == "nt"

    tree_command = shutil.which("tree")

    if tree_command is None:
        raise RuntimeError(
            "The Linux 'tree' command was not found."
        )
    ##endof:  if tree_command is None

    return _run_tree_command(
        [
            tree_command,
            "-a",
            "-f",
            ".",
        ]
    )
##endof:  _run_linux_tree(...)


def get_filtered_tree():
    windows_error = None
    linux_error = None

    try:
        return _run_windows_tree()
    except (
        RuntimeError,
        subprocess.CalledProcessError,
    ) as error:
        windows_error = error
    ##endof:  try/except

    try:
        return _run_linux_tree()
    except (
        RuntimeError,
        subprocess.CalledProcessError,
    ) as error:
        linux_error = error
    ##endof:  try/except

    raise RuntimeError(
        "Only trying Windows and Linux tree commands.\n"
        f"Windows attempt: {windows_error}\n"
        f"Linux attempt: {linux_error}"
    ) from linux_error
##endof:  get_filtered_tree(...)


try:
    tree_output = get_filtered_tree()
    print(tree_output)
except RuntimeError as error:
    print(error)
    print(
        "\nThe external tree utility is optional. "
        "MMJL assertions above are the pass/fail checks."
    )
##endof:  try/except
